---
**标题:** 张量并行 (Tensor Parallelism, TP)

**分类:** tensor-parallelism

**难度:** 中级

**预计时间:** 45 分钟

---

## 概述

当模型的某一层太大，无法放入单个 GPU 时，**张量并行 (Tensor Parallelism, TP)** 会将每个权重矩阵拆分到多个 GPU 上。

本 notebook 通过实际的张量示例，逐步构建这一思想：

1. 按列拆分矩阵 → **列并行 (Column-Parallel)**
2. 按行拆分矩阵 → **行并行 (Row-Parallel)**
3. 将两者组合成**共轭对 (Conjugate Pair)**（每个块只需一次 AllReduce）
4. 应用于 **MLP** 和**自注意力 (Self-Attention)**

### 前置知识
- PyTorch 基础（张量操作、矩阵乘法）
- [00-gpu-communication/](../00-gpu-communication/) — 尤其是 **AllReduce**（每个 GPU 将自己的数据发给所有人，最终每个 GPU 都拿到所有数据的总和）和 **AllGather**（每个 GPU 将自己的数据发出，最终每个 GPU 都拿到所有数据的拼接）

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from mp_tutorial.viz import show_matrix, show_matrices_row, draw_tensor_split, GPU_COLORS
from mp_tutorial.distributed import simulate_allreduce, simulate_allgather, check_gpu_env

check_gpu_env()
torch.manual_seed(42)

%matplotlib inline

---
## 1. 最简单的情况：一个线性层

**线性层**是神经网络最基本的构建模块 — 它计算 $Y = X \cdot W$，其中 $X$ 是输入，$W$ 是**权重矩阵**（一组可学习的数字）。训练模型就是在调整这些权重值。

让我们从可以完整看到的小张量开始。

In [ ]:
# 一个小型线性层：X (2×4) @ W (4×6) = Y (2×6)
X = torch.tensor([[1., 2., 3., 4.],
                   [5., 6., 7., 8.]])

W = torch.tensor([[1., 0., 2., 1., 0., 1.],
                   [0., 1., 1., 0., 2., 0.],
                   [1., 1., 0., 1., 1., 2.],
                   [2., 0., 1., 0., 1., 1.]])

Y = X @ W

fig, axes = plt.subplots(1, 3, figsize=(14, 2.5))
show_matrix(X, ax=axes[0], title="X（输入）", cmap="Blues")
show_matrix(W, ax=axes[1], title="W（权重）", cmap="Greens")
show_matrix(Y, ax=axes[2], title="Y = X @ W（输出）", cmap="YlOrRd")
fig.suptitle("标准线性层 - 全部在单个 GPU 上", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

现在问题来了：如果 $W$ 太大，一个 GPU 放不下怎么办？我们需要**拆分**它。

---
## 2. 列并行 (Column-Parallel)：按列拆分 $W$

将 $W$ 的列均匀分配到各个 GPU 上。每个 GPU 获得完整的输入 $X$，并使用自己那部分 $W$ 进行计算。

In [ ]:
# 将 W 拆分为 3 个列块（模拟 3 个 GPU）
num_gpus = 3
W_chunks = W.chunk(num_gpus, dim=1)  # 沿列方向拆分

# 可视化拆分结果
fig = show_matrices_row(
    W_chunks,
    titles=["W₀", "W₁", "W₂"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="第 1 步：按列拆分 W → 每个 GPU 获得 2 列",
    cmap="Greens"
)
plt.show()

In [ ]:
# 每个 GPU 计算：Y_i = X @ W_i（无需通信！）
Y_parts = [X @ W_i for W_i in W_chunks]

fig = show_matrices_row(
    Y_parts,
    titles=["Y₀ = X @ W₀", "Y₁ = X @ W₁", "Y₂ = X @ W₂"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="第 2 步：每个 GPU 执行本地矩阵乘法（无需通信！）"
)
plt.show()

In [ ]:
# 拼接得到完整输出
Y_column = torch.cat(Y_parts, dim=-1)

fig, axes = plt.subplots(1, 2, figsize=(12, 2.5))
show_matrix(Y_column, ax=axes[0], title="拼接后的 Y₀|Y₁|Y₂")
show_matrix(Y,        ax=axes[1], title="原始 Y = X @ W")
fig.suptitle("第 3 步：拼接 → 与单 GPU 结果完全一致",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_column, Y)
print("列并行结果匹配！")

**关键洞察：** 列并行的前向传播需要**零通信**。每个 GPU 独立计算其输出列。

---
## 3. 行并行 (Row-Parallel)：按行拆分 $W$

现在我们沿行方向拆分 $W$。这意味着我们还需要沿最后一个维度拆分**输入** $X$。

与列并行不同，这里每个 GPU 产生的是一个**部分和** (partial sum)，形状与完整输出相同。要得到最终结果，我们需要 **AllReduce** — 一种集体通信操作，每个 GPU 将自己的部分结果发给所有其他 GPU，最终每个 GPU 都拿到总和。

In [ ]:
# 按行拆分 W，按列拆分 X（最后一个维度），这次使用 2 个 GPU
num_gpus = 2
W_chunks = W.chunk(num_gpus, dim=0)  # 沿行方向拆分 W
X_chunks = X.chunk(num_gpus, dim=1)  # 沿列方向拆分 X

fig, axes = plt.subplots(2, 2, figsize=(10, 5))
show_matrix(X_chunks[0], ax=axes[0, 0], title="X₀（前 2 列）", gpu_label="GPU 0", cmap="Blues")
show_matrix(W_chunks[0], ax=axes[0, 1], title="W₀（前 2 行）", gpu_label="GPU 0", cmap="Greens")
show_matrix(X_chunks[1], ax=axes[1, 0], title="X₁（后 2 列）",  gpu_label="GPU 1", cmap="Blues")
show_matrix(W_chunks[1], ax=axes[1, 1], title="W₁（后 2 行）",  gpu_label="GPU 1", cmap="Greens")
fig.suptitle("第 1 步：按行拆分 W，相应地拆分 X", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# 每个 GPU 计算一个部分结果：Y_i = X_i @ W_i
Y_parts = [X_i @ W_i for X_i, W_i in zip(X_chunks, W_chunks)]

fig = show_matrices_row(
    Y_parts,
    titles=["Y₀ = X₀ @ W₀（部分和）", "Y₁ = X₁ @ W₁（部分和）"],
    gpu_labels=["GPU 0", "GPU 1"],
    suptitle="第 2 步：每个 GPU 计算一个部分和（形状相同！）"
)
plt.show()

print(f"注意：两个部分结果的形状都是 {Y_parts[0].shape} — 与完整输出相同。")
print("它们是需要相加的部分和 (partial sum)。")

In [ ]:
# AllReduce：对所有 GPU 的部分结果求和
Y_allreduced = simulate_allreduce(Y_parts)

fig, axes = plt.subplots(1, 3, figsize=(16, 2.5))
show_matrix(Y_parts[0],      ax=axes[0], title="Y₀（部分和）", gpu_label="GPU 0")
show_matrix(Y_parts[1],      ax=axes[1], title="Y₁（部分和）", gpu_label="GPU 1")
show_matrix(Y_allreduced[0], ax=axes[2], title="Y₀ + Y₁ = Y (AllReduce)")

# 在图之间添加 "+" 和 "="
fig.text(0.345, 0.45, "+", fontsize=24, fontweight="bold", ha="center")
fig.text(0.67,  0.45, "→ AllReduce →", fontsize=12, fontweight="bold", ha="center",
         color="#C44E52")
fig.suptitle("第 3 步：AllReduce 对部分和求和 → 得到完整结果", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_allreduced[0], Y)
print("行并行结果匹配！")

**关键洞察：** 行并行需要一次 **AllReduce（求和）** 来合并部分结果。

---
## 4. 并排对比

让我们在一张图中可视化两种策略，直观对比它们的区别。

In [ ]:
def draw_tp_comparison():
    """列并行与行并行的并排对比。"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

    for ax, mode in [(ax1, "column"), (ax2, "row")]:
        ax.set_xlim(0, 10)
        ax.set_ylim(0, 10)
        ax.axis("off")

        if mode == "column":
            ax.set_title("列并行 (Column-Parallel)", fontsize=14, fontweight="bold",
                        color=GPU_COLORS[0])
            steps = [
                (5, 9.0, "X（完整，已复制）", "#888"),
                (3, 7.0, "X @ W₀", GPU_COLORS[0]),
                (7, 7.0, "X @ W₁", GPU_COLORS[1]),
                (5, 5.0, "无需通信", "#55A868"),
                (5, 3.0, "Cat([Y₀, Y₁]) = Y", "#888"),
            ]
        else:
            ax.set_title("行并行 (Row-Parallel)", fontsize=14, fontweight="bold",
                        color=GPU_COLORS[1])
            steps = [
                (5, 9.0, "X 拆分 → X₀, X₁", "#888"),
                (3, 7.0, "X₀ @ W₀", GPU_COLORS[0]),
                (7, 7.0, "X₁ @ W₁", GPU_COLORS[1]),
                (5, 5.0, "AllReduce（求和）", "#C44E52"),
                (5, 3.0, "Y₀ + Y₁ = Y", "#888"),
            ]

        for x, y, text, color in steps:
            box = mpatches.FancyBboxPatch(
                (x - 2.5, y - 0.55), 5.0, 1.1,
                boxstyle="round,pad=0.15", facecolor=color, alpha=0.15,
                edgecolor=color, linewidth=2
            )
            ax.add_patch(box)
            ax.text(x, y, text, ha="center", va="center",
                    fontsize=11, fontweight="bold")

        # 箭头
        arrows = [(5, 8.4, 3, 7.6), (5, 8.4, 7, 7.6),
                  (3, 6.4, 5, 5.6), (7, 6.4, 5, 5.6),
                  (5, 4.4, 5, 3.6)]
        for x1, y1, x2, y2 in arrows:
            ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                       arrowprops=dict(arrowstyle="->", lw=1.5, color="#555"))

    fig.tight_layout()
    return fig

fig = draw_tp_comparison()
plt.show()

| | 列并行 (Column-Parallel) | 行并行 (Row-Parallel) |
|---|---|---|
| **W 拆分方式** | 按列 | 按行 |
| **X** | 复制（完整） | 沿最后一个维度拆分 |
| **每个 GPU 的输出** | Y 的列切片 | 部分和（与 Y 形状相同） |
| **通信** | 无（或 AllGather） | **AllReduce**（求和） |

---
## 5. 共轭对 (Conjugate Pair)：将两种方式组合用于 MLP

Transformer 的 **MLP（多层感知机）块**是每个 Transformer 层内部的一个小型前馈网络 — 它包含两个线性层，中间夹一个激活函数：

$$\text{MLP}(X) = \text{GeLU}(X \cdot W_1) \cdot W_2$$

**GeLU**（高斯误差线性单元）是一种激活函数 — 类似 ReLU 但更平滑。它引入非线性，使网络能学习复杂模式。对 TP 而言关键的一点是：GeLU 是逐元素操作，所以每个 GPU 可以独立地对自己的切片应用它。

Megatron 的技巧：对 $W_1$ 使用**列并行**，对 $W_2$ 使用**行并行**。列并行的输出恰好以正确的方式拆分，可以直接输入到行并行中——**两者之间无需通信！**

让我们用实际数字来追踪整个过程。

In [ ]:
# MLP: hidden=4 → intermediate=6 → hidden=4，使用 3 个 GPU
torch.manual_seed(7)
X  = (torch.randn(2, 4) * 2).round() / 2   # 较整的数字
W1 = (torch.randn(4, 6) * 2).round() / 2
W2 = (torch.randn(6, 4) * 2).round() / 2

# 参考结果
Y_ref = F.gelu(X @ W1) @ W2

print("形状：X(2×4) → W1(4×6) → GeLU → W2(6×4) → Y(2×4)")
print(f"3 个 GPU：每个获得 W1 的 2 列、W2 的 2 行")

In [ ]:
num_gpus = 3

# 第 1 步：对 W1 进行列并行拆分
W1_chunks = W1.chunk(num_gpus, dim=1)

fig, axes = plt.subplots(1, 4, figsize=(16, 3),
                         gridspec_kw={"width_ratios": [4, 2, 2, 2]})
show_matrix(W1, ax=axes[0], title="W₁（完整）", cmap="Greens")
for i, chunk in enumerate(W1_chunks):
    show_matrix(chunk, ax=axes[i+1], title=f"W₁ᵢ", gpu_label=f"GPU {i}", cmap="Greens")
fig.suptitle("MLP 第 1 步：按列拆分 W₁", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# 第 2 步：每个 GPU 计算 X @ W1_i，然后在本地应用 GeLU
H_parts = [F.gelu(X @ W1_i) for W1_i in W1_chunks]

fig = show_matrices_row(
    H_parts,
    titles=["GeLU(X @ W₁₀)", "GeLU(X @ W₁₁)", "GeLU(X @ W₁₂)"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="MLP 第 2 步：每个 GPU 在本地执行矩阵乘法 + GeLU（无需通信）",
    cmap="Purples"
)
plt.show()

print("每个 GPU 有一个 (2×2) 的隐藏激活 — 中间层的一个切片。")

In [ ]:
# 第 3 步：对 W2 进行行并行拆分 — 按行拆分 W2
# 列并行的输出已经按正确方式拆分好了！
W2_chunks = W2.chunk(num_gpus, dim=0)

fig, axes = plt.subplots(1, 4, figsize=(16, 3),
                         gridspec_kw={"width_ratios": [4, 2, 2, 2]})
show_matrix(W2, ax=axes[0], title="W₂（完整）", cmap="Greens")
for i, chunk in enumerate(W2_chunks):
    show_matrix(chunk, ax=axes[i+1], title=f"W₂ᵢ", gpu_label=f"GPU {i}", cmap="Greens")
fig.suptitle("MLP 第 3 步：按行拆分 W₂（与列拆分匹配！）",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
# 第 4 步：每个 GPU 计算 H_i @ W2_i → 部分输出（与最终 Y 形状相同）
Y_parts = [H_i @ W2_i for H_i, W2_i in zip(H_parts, W2_chunks)]

fig = show_matrices_row(
    Y_parts,
    titles=["H₀ @ W₂₀", "H₁ @ W₂₁", "H₂ @ W₂₂"],
    gpu_labels=["GPU 0", "GPU 1", "GPU 2"],
    suptitle="MLP 第 4 步：每个 GPU 有一个输出的部分和"
)
plt.show()

print("三个结果都是 (2×4) — 与最终输出形状相同。它们需要被求和。")

In [ ]:
# 第 5 步：AllReduce — 整个 MLP 中唯一的通信！
Y_final = simulate_allreduce(Y_parts)

fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
show_matrix(Y_final[0], ax=axes[0], title="AllReduce 结果")
show_matrix(Y_ref,      ax=axes[1], title="参考值（单 GPU MLP）")
fig.suptitle("MLP 第 5 步：AllReduce 对 3 个部分和求和 → 与参考值匹配",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_final[0], Y_ref, atol=1e-5)
print("TP MLP 结果匹配！整个 MLP 块只需 1 次 AllReduce。")

### 为什么这种方法有效？

列并行的输出自然地衔接行并行的输入：

| 步骤 | 发生了什么 | 通信 |
|------|-----------|------|
| W₁ 列并行 | 每个 GPU 获得 W₁ 的列，使用完整 X 计算 | 无 |
| GeLU | 在每个 GPU 的切片上本地应用 | 无 |
| W₂ 行并行 | 每个 GPU 的切片正好是正确的"行"输入 | 无 |
| AllReduce | 对部分输出求和得到最终 Y | **1 次 AllReduce** |

这就是**共轭对 (Conjugate Pair)** — Megatron 风格张量并行的基石。

> **难度提升提示：** 下一节将 TP 应用于自注意力机制，涉及 Q/K/V 投影和多头拆分。如果你对注意力机制还不熟悉，只需关注核心思想：注意力头是*独立的*，因此可以自然地拆分到各 GPU 上——无需额外通信。

---
## 6. 自注意力 (Self-Attention)：相同模式，不同层

**多头自注意力**是 Transformer 层的另一个核心组件（与 MLP 并列）。它让模型能够查看序列中的所有位置，判断哪些位置之间彼此相关。

它通过将输入投影到三组向量来工作 — **Q（查询）**、**K（键）**、**V（值）** — 使用不同的权重矩阵。位置之间的注意力分数计算为 $\text{softmax}(QK^T / \sqrt{d})$，然后用来加权值 $V$。

"多头"意味着这个过程用不同的学习投影（称为**头**）并行执行多次。每个头可以关注不同的模式。由于各头相互独立，它们非常适合 TP — 每个 GPU 处理一部分头。

In [ ]:
def draw_attention_tp(num_heads=8, num_gpus=4):
    """可视化注意力头在 GPU 之间的分布。"""
    heads_per_gpu = num_heads // num_gpus

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.set_xlim(-0.5, num_heads + 3)
    ax.set_ylim(-1, 4.5)
    ax.axis("off")

    # 将注意力头绘制为彩色方块
    for h in range(num_heads):
        gpu = h // heads_per_gpu
        color = GPU_COLORS[gpu % len(GPU_COLORS)]
        rect = mpatches.FancyBboxPatch(
            (h, 2), 0.85, 1.5,
            boxstyle="round,pad=0.1", facecolor=color, alpha=0.7,
            edgecolor="white", linewidth=2
        )
        ax.add_patch(rect)
        ax.text(h + 0.42, 2.75, f"H{h}", ha="center", va="center",
                fontsize=11, fontweight="bold", color="white")

    # GPU 标签
    for g in range(num_gpus):
        start = g * heads_per_gpu
        mid = start + heads_per_gpu / 2 - 0.08
        color = GPU_COLORS[g % len(GPU_COLORS)]
        ax.text(mid, 1.3, f"GPU {g}", ha="center", va="center",
                fontsize=10, fontweight="bold", color="white",
                bbox=dict(boxstyle="round,pad=0.3", fc=color, ec="none"))
        # 括号
        ax.plot([start, start + heads_per_gpu - 0.15],
                [1.7, 1.7], color=color, linewidth=2)

    # 标签
    ax.text(num_heads / 2 - 0.08, 4.1,
            "Q, K, V 投影：列并行（每个 GPU 获得其对应的头）",
            ha="center", fontsize=11, fontstyle="italic")
    ax.text(num_heads / 2 - 0.08, 0.3,
            "输出投影：行并行 → AllReduce",
            ha="center", fontsize=11, fontstyle="italic", color="#C44E52")

    ax.set_title(f"{num_heads} 个注意力头分布在 {num_gpus} 个 GPU 上",
                fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

fig = draw_attention_tp(num_heads=8, num_gpus=4)
plt.show()

In [ ]:
# 模拟：4 个头，hidden=8，head_dim=2，2 个 GPU
torch.manual_seed(0)
num_heads, head_dim, hidden = 4, 2, 8
num_gpus = 2
seq_len = 3

X = (torch.randn(seq_len, hidden) * 2).round() / 2

# 完整的 Q, K, V 投影权重：(hidden, num_heads * head_dim)
Wq = torch.randn(hidden, num_heads * head_dim)
Wk = torch.randn(hidden, num_heads * head_dim)
Wv = torch.randn(hidden, num_heads * head_dim)
Wo = torch.randn(num_heads * head_dim, hidden)  # 输出投影

# --- 参考：单 GPU 注意力 ---
Q = X @ Wq  # (seq, num_heads*head_dim)
K = X @ Wk
V = X @ Wv

# 重塑为 (num_heads, seq, head_dim) 用于注意力计算
Q_heads = Q.view(seq_len, num_heads, head_dim).permute(1, 0, 2)
K_heads = K.view(seq_len, num_heads, head_dim).permute(1, 0, 2)
V_heads = V.view(seq_len, num_heads, head_dim).permute(1, 0, 2)

attn = torch.softmax(Q_heads @ K_heads.transpose(-2, -1) / head_dim**0.5, dim=-1)
attn_out = (attn @ V_heads).permute(1, 0, 2).reshape(seq_len, num_heads * head_dim)
Y_ref = attn_out @ Wo
print(f"参考注意力输出形状：{Y_ref.shape}")

In [ ]:
# --- TP 注意力：将头分布到各 GPU ---
heads_per_gpu = num_heads // num_gpus  # 每个 GPU 2 个头

# 列并行：按列拆分 Wq, Wk, Wv（每个 GPU 获得其对应头的权重）
Wq_chunks = Wq.chunk(num_gpus, dim=1)
Wk_chunks = Wk.chunk(num_gpus, dim=1)
Wv_chunks = Wv.chunk(num_gpus, dim=1)
# 行并行：按行拆分 Wo
Wo_chunks = Wo.chunk(num_gpus, dim=0)

Y_parts = []
for g in range(num_gpus):
    # 每个 GPU 计算其子集的头
    Q_g = X @ Wq_chunks[g]  # (seq, heads_per_gpu * head_dim)
    K_g = X @ Wk_chunks[g]
    V_g = X @ Wv_chunks[g]

    Q_h = Q_g.view(seq_len, heads_per_gpu, head_dim).permute(1, 0, 2)
    K_h = K_g.view(seq_len, heads_per_gpu, head_dim).permute(1, 0, 2)
    V_h = V_g.view(seq_len, heads_per_gpu, head_dim).permute(1, 0, 2)

    attn_g = torch.softmax(Q_h @ K_h.transpose(-2, -1) / head_dim**0.5, dim=-1)
    out_g = (attn_g @ V_h).permute(1, 0, 2).reshape(seq_len, heads_per_gpu * head_dim)

    # 行并行输出投影 → 部分结果
    Y_parts.append(out_g @ Wo_chunks[g])

fig = show_matrices_row(
    Y_parts,
    titles=["头 0-1 → 部分 Y", "头 2-3 → 部分 Y"],
    gpu_labels=["GPU 0", "GPU 1"],
    suptitle="每个 GPU 处理其注意力头 → 部分输出"
)
plt.show()

In [ ]:
# AllReduce 得到最终结果
Y_tp = simulate_allreduce(Y_parts)

fig, axes = plt.subplots(1, 2, figsize=(10, 2.5))
show_matrix(Y_tp[0], ax=axes[0], title="TP 注意力（AllReduce 后）")
show_matrix(Y_ref,   ax=axes[1], title="参考值（单 GPU）")
fig.suptitle("AllReduce → 与单 GPU 注意力结果匹配", fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

assert torch.allclose(Y_tp[0], Y_ref, atol=1e-5)
print("TP 注意力结果匹配！相同模式：列并行 → 本地计算 → 行并行 → AllReduce")

---
## 7. 汇总：一个 Transformer 层

每个 Transformer 层包含一个注意力块和一个 MLP 块。使用张量并行时，每个块恰好需要**一次 AllReduce**。

In [ ]:
def draw_transformer_tp_flow():
    """可视化一个 TP Transformer 层中的通信模式。"""
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.axis("off")

    blocks = [
        # (x, y, w, h, text, color, comm_label)
        (5, 11,  8, 0.8, "输入 X（已复制）", "#888", None),
        (5, 9.5, 8, 0.8, "Q,K,V：列并行", GPU_COLORS[0], "无通信"),
        (5, 8.0, 8, 0.8, "注意力（每个 GPU 本地计算）", GPU_COLORS[2], "无通信"),
        (5, 6.5, 8, 0.8, "输出投影：行并行", GPU_COLORS[1], None),
        (5, 5.0, 8, 0.8, "★ AllReduce #1", "#C44E52", None),
        (5, 3.5, 8, 0.8, "W₁：列并行 + GeLU", GPU_COLORS[0], "无通信"),
        (5, 2.0, 8, 0.8, "W₂：行并行", GPU_COLORS[1], None),
        (5, 0.5, 8, 0.8, "★ AllReduce #2", "#C44E52", None),
    ]

    section_labels = [
        (9.8, 8.0, "自注意力"),
        (9.8, 2.7, "MLP"),
    ]

    for x, y, w, h, text, color, comm in blocks:
        box = mpatches.FancyBboxPatch(
            (x - w/2, y - h/2), w, h,
            boxstyle="round,pad=0.1", facecolor=color, alpha=0.15,
            edgecolor=color, linewidth=2
        )
        ax.add_patch(box)
        fs = 12 if "★" in text else 10
        fw = "bold" if "★" in text else "bold"
        ax.text(x, y, text, ha="center", va="center", fontsize=fs, fontweight=fw)
        if comm:
            ax.text(x + w/2 - 0.2, y, comm, ha="left", va="center",
                    fontsize=8, fontstyle="italic", color="#55A868")

    # 箭头
    for i in range(len(blocks) - 1):
        y1 = blocks[i][1] - blocks[i][3]/2
        y2 = blocks[i+1][1] + blocks[i+1][3]/2
        ax.annotate("", xy=(5, y2), xytext=(5, y1),
                   arrowprops=dict(arrowstyle="->", lw=1.5, color="#555"))

    for x, y, text in section_labels:
        ax.text(x, y, text, ha="center", va="center", fontsize=10,
                fontstyle="italic", color="#666",
                bbox=dict(boxstyle="round", fc="#f0f0f0", ec="#ccc"))

    ax.set_title("使用张量并行的一个 Transformer 层\n"
                 "总通信量：仅 2 次 AllReduce 操作",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    return fig

fig = draw_transformer_tp_flow()
plt.show()

---
## 8. 在大语言模型中的应用

In [ ]:
def draw_tp_in_practice():
    """可视化为什么 TP 限制在单节点内。"""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # 左图：带宽对比
    ax = axes[0]
    labels = ["NVLink\n（节点内）", "InfiniBand\n（节点间）"]
    bw = [600, 50]
    colors = ["#55A868", "#C44E52"]
    bars = ax.barh(labels, bw, color=colors, height=0.5, edgecolor="white")
    for bar, val in zip(bars, bw):
        ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
                f"{val} GB/s", va="center", fontsize=12, fontweight="bold")
    ax.set_xlim(0, 750)
    ax.set_xlabel("每个 GPU 的带宽", fontsize=10)
    ax.set_title("为什么 TP 只在节点内使用", fontsize=12, fontweight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # 右图：模型配置
    ax = axes[1]
    ax.axis("off")
    table_data = [
        ["GPT-3",       "175B", "8",  "8× A100, NVLink"],
        ["LLaMA 2 70B", "70B",  "8",  "8× A100-80GB"],
        ["MT-NLG",      "530B", "8",  "+ 35 路 PP, 64 路 DP"],
        ["PaLM",        "540B", "8",  "TPU 主机内 8 路"],
    ]
    table = ax.table(
        cellText=table_data,
        colLabels=["模型", "参数量", "TP 度", "备注"],
        loc="center", cellLoc="center"
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.0, 1.6)
    for (r, c), cell in table.get_celld().items():
        if r == 0:
            cell.set_facecolor("#4C72B0")
            cell.set_text_props(color="white", fontweight="bold")
        else:
            cell.set_facecolor("#f7f7f7" if r % 2 else "white")
    ax.set_title("实际应用中的 TP：度数始终为 8", fontsize=12, fontweight="bold")

    fig.tight_layout()
    return fig

fig = draw_tp_in_practice()
plt.show()

TP 的并行度几乎总是 **8** — 受限于通过 **NVLink**（NVIDIA 的高速 GPU 间互联，约 600 GB/s）连接的单节点内的 GPU。跨节点使用 **InfiniBand**（约 50 GB/s）— 慢 12 倍，无法满足 TP 频繁 AllReduce 的需求。

要扩展到节点之外，需要将 TP 与**流水线并行** (Pipeline Parallelism, PP — 按层将模型拆分到不同节点) 和**数据并行** (Data Parallelism, DP — 每个节点处理不同的数据批次) 结合使用。

---
## 9. 需要 GPU：真正的多 GPU 张量并行

> **需要 GPU** — 请在远程机器上运行此单元格（a multi-GPU machine，4 块 GPU）。

In [ ]:
# [需要 GPU]
# 使用 torch.distributed + NCCL 实现真正的多 GPU 张量并行
# 在远程机器上运行：a multi-GPU machine（4 块 GPU）

import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn.functional as F
import os

def tp_mlp_worker(rank, world_size, X_shared, W1_shared, W2_shared, results):
    """张量并行 MLP 中单个 GPU 的工作函数。"""
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "29500"
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

    device = torch.device(f"cuda:{rank}")
    X = X_shared.to(device)

    # 列并行 W1
    chunk_size = W1_shared.shape[1] // world_size
    W1_local = W1_shared[:, rank * chunk_size:(rank + 1) * chunk_size].to(device)

    # 行并行 W2
    row_chunk = W2_shared.shape[0] // world_size
    W2_local = W2_shared[rank * row_chunk:(rank + 1) * row_chunk, :].to(device)

    # 前向传播：列并行 → GeLU → 行并行 → AllReduce
    hidden = F.gelu(X @ W1_local)
    output = hidden @ W2_local
    dist.all_reduce(output, op=dist.ReduceOp.SUM)

    if rank == 0:
        results["output"] = output.cpu()
    dist.destroy_process_group()

if torch.cuda.is_available() and torch.cuda.device_count() >= 4:
    world_size = 4
    hidden, intermediate = 64, 256
    X = torch.randn(2, 8, hidden)
    W1 = torch.randn(hidden, intermediate)
    W2 = torch.randn(intermediate, hidden)

    Y_ref = F.gelu(X @ W1) @ W2

    manager = mp.Manager()
    results = manager.dict()
    mp.spawn(tp_mlp_worker, args=(world_size, X, W1, W2, results), nprocs=world_size)

    Y_tp = results["output"]
    print(f"最大误差：{(Y_tp - Y_ref).abs().max().item():.2e}")
    print("真正的多 GPU TP MLP 与参考值匹配！")
else:
    print("跳过：需要 4 块 CUDA GPU。请通过 a multi-GPU machine 连接")

---
## 10. Megatron-LM 参考实现

Megatron-LM 是张量并行的标准生产实现。两个关键类：

In [ ]:
from mp_tutorial.formatting import code_reference

code_reference(
    code="""
class ColumnParallelLinear(torch.nn.Module):
    \"\"\"Y = XA + b, with A split by columns: A = [A_1, ..., A_p]\"\"\"

    def __init__(self, input_size, output_size, *, gather_output=True, ...):
        self.output_size_per_partition = divide(output_size, world_size)
        self.weight = Parameter(torch.empty(
            self.output_size_per_partition, input_size))  # transposed!

    def forward(self, input_):
        # Identity in fwd, AllReduce in bwd
        input_parallel = copy_to_tensor_model_parallel_region(input_)
        output_parallel = F.linear(input_parallel, self.weight, self.bias)
        if self.gather_output:
            output = gather_from_tensor_model_parallel_region(output_parallel)
        else:
            output = output_parallel  # keep split for next layer
        return output
""",
    source="Megatron-LM",
    filepath="megatron/core/tensor_parallel/layers.py"
)

In [ ]:
code_reference(
    code="""
class RowParallelLinear(torch.nn.Module):
    \"\"\"Y = XA + b, with A split by rows and X by columns\"\"\"

    def __init__(self, input_size, output_size, *, input_is_parallel=False, ...):
        self.input_size_per_partition = divide(input_size, world_size)
        self.weight = Parameter(torch.empty(
            output_size, self.input_size_per_partition))

    def forward(self, input_):
        if self.input_is_parallel:
            input_parallel = input_  # already split from column-parallel
        else:
            input_parallel = scatter_to_tensor_model_parallel_region(input_)
        output_parallel = F.linear(input_parallel, self.weight)
        # AllReduce in fwd, identity in bwd
        output_ = reduce_from_tensor_model_parallel_region(output_parallel)
        return output_ + self.bias if self.bias is not None else output_
""",
    source="Megatron-LM",
    filepath="megatron/core/tensor_parallel/layers.py"
)

`mappings.py` 将通信原语定义为自定义 autograd 函数 — 每个都是一个**共轭对 (conjugate pair)**，确保梯度流的正确性：

| 前向传播 | 反向传播 | 使用者 |
|---------|---------|--------|
| Identity | AllReduce | `CopyToModelParallelRegion`（列并行输入） |
| AllReduce | Identity | `ReduceFromModelParallelRegion`（行并行输出） |
| Split | AllGather | `ScatterToModelParallelRegion` |
| AllGather | Split | `GatherFromModelParallelRegion` |

---
## 总结与延伸阅读

### 核心要点

1. **列并行 (Column-Parallel)** 按列拆分 $W$ → 前向传播无需通信
2. **行并行 (Row-Parallel)** 按行拆分 $W$ → 需要 AllReduce 对部分和求和
3. **共轭对 (Conjugate Pair)**（列 → 行）每个块只需 **1 次 AllReduce**
4. 每个 Transformer 层：注意力 1 次 AllReduce + MLP 1 次 = **共 2 次**
5. TP 并行度约为 **8**（限制在通过 NVLink 连接的单节点内）
6. 自注意力天然适配：每个 GPU 获得一部分**注意力头**

### 延伸阅读

- [Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism](https://arxiv.org/abs/1909.08053)
- [Efficient Large-Scale Language Model Training on GPU Clusters](https://arxiv.org/abs/2104.04473) — 3D 并行 (TP + PP + DP)
- [PyTorch Distributed 文档](https://pytorch.org/docs/stable/distributed.html)
- [NVIDIA Megatron-Core](https://github.com/NVIDIA/Megatron-LM)
- 下一篇：[03-pipeline-parallelism/](../03-pipeline-parallelism/) — 跨 GPU 按层拆分